# Tahap 2: Pelatihan Model Klasifikasi Terawasi Multilabel (Supervised Multilabel)

Notebook ini bertujuan untuk mengimplementasikan dan melatih model tahap kedua dalam alur hibrida penelitian rekomendasi tanaman multilabel. 

**Alur Proses:**
1. Memuat dataset dan melakukan data preprocessing (Standard Scaling & PCA 5-Komponen).
2. Memuat model klasterisasi tahap pertama (K-Means & GMM) untuk memetakan profile lahan ke klaster dan mengidentifikasi tanaman yang kompatibel.
3. Membentuk label target biner multilabel (multi-hot encoding) untuk ke-22 kelas tanaman.
4. Melatih model klasifikasi terawasi multilabel (seperti Random Forest via `MultiOutputClassifier`).
5. Mengevaluasi performa prediksi menggunakan metrik **Subset Accuracy (Exact Match Ratio)** dan **F1-Score (Micro & Macro)**.

In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.preprocessing import StandardScaler, MultiLabelBinarizer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

print("Library berhasil diimpor.")

Library berhasil diimpor.


## 1. Memuat dan Mempersiapkan Dataset

Kita memuat dataset `crop_recommendation.csv` dan memprosesnya dengan Standard Scaling serta reduksi dimensi PCA 5-Komponen agar selaras dengan model klasterisasi tahap pertama.

In [2]:
# Load dataset
df = pd.read_csv('crop_recommendation.csv')
features = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']
X = df[features]
y_true = df['label']

# Load scaler & PCA jika ada, atau buat baru untuk memastikan konsistensi
try:
    scaler = joblib.load('scaler.joblib')
    pca_5 = joblib.load('pca_5.joblib')
    X_scaled = scaler.transform(X)
    X_pca_5 = pca_5.transform(X_scaled)
    print("Berhasil memuat scaler.joblib dan pca_5.joblib dari disk.")
except Exception as e:
    print("Model joblib tidak ditemukan atau terjadi kesalahan. Membuat baru...")
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    pca_5 = PCA(n_components=5)
    X_pca_5 = pca_5.fit_transform(X_scaled)

print(f"Dimensi data setelah Scaling & PCA 5D: {X_pca_5.shape}")

Berhasil memuat scaler.joblib dan pca_5.joblib dari disk.
Dimensi data setelah Scaling & PCA 5D: (2200, 5)


## 2. Formulasi Target Multilabel dari Model K-Means (Hard Clustering)

Pada tahap ini, kita memuat model K-Means ($K=22$) yang telah dilatih, lalu memetakan klaster hasil K-Means menjadi representasi label tanaman multilabel (seluruh varietas yang masuk ke dalam klaster tersebut).

In [3]:
# Load model K-Means jika ada, atau latih yang baru
try:
    kmeans = joblib.load('kmeans_model.joblib')
    print("Berhasil memuat kmeans_model.joblib.")
except:
    print("kmeans_model.joblib tidak ditemukan. Melatih model K-Means baru (K=22)...")
    kmeans = KMeans(n_clusters=22, init='k-means++', random_state=42, n_init=10)
    kmeans.fit(X_pca_5)

# Prediksi label klaster
df['KMeans_Cluster'] = kmeans.predict(X_pca_5)

# Petakan setiap klaster ke daftar tanaman unik yang ada di dalamnya
kmeans_cluster_crops = df.groupby('KMeans_Cluster')['label'].apply(set).to_dict()

print("Contoh pemetaan 5 klaster pertama K-Means ke tanaman:")
for c in sorted(list(kmeans_cluster_crops.keys()))[:5]:
    print(f"Klaster {c}: {list(kmeans_cluster_crops[c])}")

# Bentuk representasi target multilabel
y_multilabel_kmeans = df['KMeans_Cluster'].map(kmeans_cluster_crops).tolist()

# Transformasikan ke format binary indicator matrix (multi-hot encoding)
mlb_kmeans = MultiLabelBinarizer()
y_bin_kmeans = mlb_kmeans.fit_transform(y_multilabel_kmeans)
print(f"\nDimensi target biner multilabel K-Means: {y_bin_kmeans.shape}")

Berhasil memuat kmeans_model.joblib.
Contoh pemetaan 5 klaster pertama K-Means ke tanaman:
Klaster 0: ['orange', 'coffee', 'pomegranate', 'maize', 'jute', 'pigeonpeas']
Klaster 1: ['grapes', 'apple']
Klaster 2: ['banana', 'papaya']
Klaster 3: ['papaya', 'coconut', 'pigeonpeas']
Klaster 4: ['chickpea', 'kidneybeans', 'lentil', 'pigeonpeas']

Dimensi target biner multilabel K-Means: (2200, 22)


## 3. Formulasi Target Multilabel dari Model GMM (Soft Clustering)

Sama halnya dengan K-Means, kita memetakan komponen Gaussian Mixture Model (n=22) ke kelompok tanaman yang ada di dalamnya untuk dijadikan target multilabel.

In [4]:
# Load model GMM jika ada, atau latih yang baru
try:
    gmm = joblib.load('gmm_model.joblib')
    print("Berhasil memuat gmm_model.joblib.")
except:
    print("gmm_model.joblib tidak ditemukan. Melatih model GMM baru (n_components=22)...")
    gmm = GaussianMixture(n_components=22, random_state=42)
    gmm.fit(X_pca_5)

# Prediksi label klaster
df['GMM_Cluster'] = gmm.predict(X_pca_5)

# Petakan setiap klaster ke daftar tanaman unik yang ada di dalamnya
gmm_cluster_crops = df.groupby('GMM_Cluster')['label'].apply(set).to_dict()

print("Contoh pemetaan 5 klaster pertama GMM ke tanaman:")
for c in sorted(list(gmm_cluster_crops.keys()))[:5]:
    print(f"Klaster {c}: {list(gmm_cluster_crops[c])}")

# Bentuk representasi target multilabel
y_multilabel_gmm = df['GMM_Cluster'].map(gmm_cluster_crops).tolist()

# Transformasikan ke format binary indicator matrix (multi-hot encoding)
mlb_gmm = MultiLabelBinarizer()
y_bin_gmm = mlb_gmm.fit_transform(y_multilabel_gmm)
print(f"\nDimensi target biner multilabel GMM: {y_bin_gmm.shape}")

Berhasil memuat gmm_model.joblib.
Contoh pemetaan 5 klaster pertama GMM ke tanaman:
Klaster 0: ['mungbean', 'blackgram', 'mothbeans', 'lentil', 'pigeonpeas']
Klaster 1: ['papaya', 'rice', 'jute']
Klaster 2: ['apple']
Klaster 3: ['cotton']
Klaster 4: ['mothbeans', 'pigeonpeas']

Dimensi target biner multilabel GMM: (2200, 22)


## 4. Pelatihan Model Supervised Multilabel berbasis K-Means

Kita membagi data menjadi set pelatihan dan pengujian (80:20), lalu melatih `MultiOutputClassifier` yang dibungkus dengan `RandomForestClassifier` untuk memprediksi rekomendasi tanaman biner dari input koordinat PCA 5D.

In [5]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X_pca_5, y_bin_kmeans, test_size=0.2, random_state=42
)

# Latih model Random Forest Multilabel
clf_kmeans = MultiOutputClassifier(RandomForestClassifier(n_estimators=100, random_state=42))
clf_kmeans.fit(X_train, y_train)

# Prediksi
y_pred_kmeans = clf_kmeans.predict(X_test)

# Hitung metrik evaluasi
sub_acc_kmeans = accuracy_score(y_test, y_pred_kmeans)
f1_micro_kmeans = f1_score(y_test, y_pred_kmeans, average='micro')
f1_macro_kmeans = f1_score(y_test, y_pred_kmeans, average='macro')

print("Hasil Evaluasi Model Multilabel Berbasis K-Means:")
print(f"Subset Accuracy (Exact Match Ratio): {sub_acc_kmeans * 100:.2f}%")
print(f"F1-Score (Micro Average)          : {f1_micro_kmeans:.4f}")
print(f"F1-Score (Macro Average)          : {f1_macro_kmeans:.4f}")

# Simpan model multilabel K-Means
joblib.dump(clf_kmeans, 'supervised_multilabel_kmeans.joblib')
joblib.dump(mlb_kmeans, 'mlb_kmeans.joblib')
print("\nModel dan binarizer berhasil disimpan.")

Hasil Evaluasi Model Multilabel Berbasis K-Means:
Subset Accuracy (Exact Match Ratio): 81.36%
F1-Score (Micro Average)          : 0.9537
F1-Score (Macro Average)          : 0.9557



Model dan binarizer berhasil disimpan.


## 5. Pelatihan Model Supervised Multilabel berbasis GMM

Dengan langkah yang sama, kita melatih pengklasifikasi multilabel menggunakan target biner yang didasarkan pada klasterisasi GMM.

In [6]:
# Split data
X_train_gm, X_test_gm, y_train_gm, y_test_gm = train_test_split(
    X_pca_5, y_bin_gmm, test_size=0.2, random_state=42
)

# Latih model Random Forest Multilabel untuk GMM
clf_gmm = MultiOutputClassifier(RandomForestClassifier(n_estimators=100, random_state=42))
clf_gmm.fit(X_train_gm, y_train_gm)

# Prediksi
y_pred_gmm = clf_gmm.predict(X_test_gm)

# Hitung metrik evaluasi
sub_acc_gmm = accuracy_score(y_test_gm, y_pred_gmm)
f1_micro_gmm = f1_score(y_test_gm, y_pred_gmm, average='micro')
f1_macro_gmm = f1_score(y_test_gm, y_pred_gmm, average='macro')

print("Hasil Evaluasi Model Multilabel Berbasis GMM:")
print(f"Subset Accuracy (Exact Match Ratio): {sub_acc_gmm * 100:.2f}%")
print(f"F1-Score (Micro Average)          : {f1_micro_gmm:.4f}")
print(f"F1-Score (Macro Average)          : {f1_macro_gmm:.4f}")

# Simpan model multilabel GMM
joblib.dump(clf_gmm, 'supervised_multilabel_gmm.joblib')
joblib.dump(mlb_gmm, 'mlb_gmm.joblib')
print("\nModel dan binarizer berhasil disimpan.")

Hasil Evaluasi Model Multilabel Berbasis GMM:
Subset Accuracy (Exact Match Ratio): 84.77%
F1-Score (Micro Average)          : 0.9404
F1-Score (Macro Average)          : 0.9489



Model dan binarizer berhasil disimpan.


## 6. Komparasi Hasil Tahap 2

Kita membandingkan performa model klasifikasi terawasi tahap kedua berdasarkan data target klaster K-Means vs GMM.

In [7]:
perbandingan_df = pd.DataFrame({
    'Model Pendekatan Tahap 1': ['K-Means (Hard Clustering)', 'GMM (Soft Clustering)'],
    'Subset Accuracy': [f"{sub_acc_kmeans * 100:.2f}%", f"{sub_acc_gmm * 100:.2f}%"],
    'F1-Score Micro': [f"{f1_micro_kmeans:.4f}", f"{f1_micro_gmm:.4f}"],
    'F1-Score Macro': [f"{f1_macro_kmeans:.4f}", f"{f1_macro_gmm:.4f}"]
})

print("Tabel Komparasi Performa Tahap 2 (Supervised Multilabel):")
display(perbandingan_df)

Tabel Komparasi Performa Tahap 2 (Supervised Multilabel):


,Model Pendekatan Tahap 1,Subset Accuracy,F1-Score Micro,F1-Score Macro
0,K-Means (Hard Clustering),81.36%,0.9537,0.9557
1,GMM (Soft Clustering),84.77%,0.9404,0.9489


## 7. Simulasi Prediksi Rekomendasi Tanaman Baru

Bagian ini menunjukkan cara memprediksi daftar tanaman alternatif untuk sebidang tanah baru dengan parameter masukan agronomi.

In [8]:
# Input lahan simulasi (N, P, K, suhu, kelembaban, ph, curah hujan)
new_land = pd.DataFrame([[90, 42, 43, 25.0, 80.0, 6.5, 150.0]], columns=features)

# Scaling dan PCA
new_land_scaled = scaler.transform(new_land)
new_land_pca = pca_5.transform(new_land_scaled)

# Prediksi rekomendasi biner menggunakan model berbasis GMM
pred_bin = clf_gmm.predict(new_land_pca)
recommended_crops = mlb_gmm.inverse_transform(pred_bin)

print("Kondisi Agronomi Input Lahan Baru:")
display(new_land)
print("\nRekomendasi Varietas Tanaman Alternatif:")
if recommended_crops[0]:
    for c in recommended_crops[0]:
        print(f"- {c.capitalize()}")
else:
    print("Tidak ada tanaman yang direkomendasikan untuk kondisi lahan ini.")

Kondisi Agronomi Input Lahan Baru:


,N,P,K,temperature,humidity,ph,rainfall
0,90,42,43,25.0,80.0,6.5,150.0



Rekomendasi Varietas Tanaman Alternatif:
- Coffee
- Jute
- Pomegranate
- Rice
